## Mongo Twitter

1. code to extract Twitter from MongoDB once Mongo is running locally. 
2. some sample in csv format
3. some sample in json format
4. more sample in jsom format

### Extract Twitter from Mongo Db 



In [ ]:
import pandas as pd
from pymongo import MongoClient
from datetime import datetime
import json

def extract_twitter_data(mongo_uri='mongodb://localhost:27017/', 
                        db_name='your_db_name', 
                        collection_name='QCPS_2',
                        sample_size=10000000):
    """
    Extracts a sample of tweets from MongoDB and organizes them into a pandas DataFrame.
    """
    # Connect to MongoDB
    client = MongoClient(mongo_uri)
    db = client[db_name]
    collection = db[collection_name]
    
    # Define our aggregation pipeline
    pipeline = [
        {"$sample": {"size": sample_size}},
        {"$project": {
            "_id": 0,
            "text": 1,
            "created_at": 1,
            "userMentionEntitiesArray": {
                "$ifNull": ["$userMentionEntitiesArray", []]  # Handle missing arrays in MongoDB
            },
            "hashtagEntitiesArray": {
                "$ifNull": ["$hashtagEntitiesArray", []]  # Handle missing arrays in MongoDB
            }
        }}
    ]
    
    print("Fetching data from MongoDB...")
    tweets_data = list(collection.aggregate(pipeline))
    
    print("Converting to DataFrame...")
    df = pd.DataFrame(tweets_data)
    
    # Convert the date field
    df['created_at'] = pd.to_datetime(df['created_at'])
    
    # Handle missing arrays - this is a safer approach than the previous version
    df['userMentionEntitiesArray'] = df['userMentionEntitiesArray'].fillna(pd.Series([[]] * len(df)))
    df['hashtagEntitiesArray'] = df['hashtagEntitiesArray'].fillna(pd.Series([[]] * len(df)))
    
    # Print information about the extracted data
    print("\nDataset Overview:")
    print(f"Number of tweets: {len(df)}")
    print(f"Date range: {df['created_at'].min()} to {df['created_at'].max()}")
    print(f"Tweets with mentions: {df['userMentionEntitiesArray'].apply(len).gt(0).sum()}")
    print(f"Tweets with hashtags: {df['hashtagEntitiesArray'].apply(len).gt(0).sum()}")
    
    return df

def save_dataframe(df, csv_path='twitter_sample.csv', pickle_path='twitter_sample.pkl'):
    """
    Saves the DataFrame both as CSV and pickle format.
    """
    print(f"\nSaving to CSV: {csv_path}")
    df.to_csv(csv_path, index=False)
    
    print(f"Saving to pickle: {pickle_path}")
    df.to_pickle(pickle_path)

def main():
    # Extract data
    df = extract_twitter_data(
        mongo_uri='mongodb://localhost:27017/',
        db_name='twitter',
        collection_name='QCPS_2',
        sample_size=10000000
    )
    
    # Save the data
    save_dataframe(df)
    
    # Display a few example rows
    print("\nFirst few rows of the dataset:")
    pd.set_option('display.max_columns', None)  # Show all columns
    print(df.head())

if __name__ == "__main__":
    main()

### Sample from csv file

created_at,text,userMentionEntitiesArray,hashtagEntitiesArray
2022-02-23 19:38:33,@NicolettaC913 Ma per me li tutti leggono Twitter cioè ci leggono ma non dovrebbe succedere .troppe cose che sanno che noi scriviamo qui.e sicuramente anche Lulù sa del casino che abbiamo creato per ieri notte oggi era troppo giù #jeru,['NicolettaC913'],['jeru']
2022-02-24 22:09:21,"RT @glisteninIights: comunque le princesses sono veramente la mia vita, in un blocco le vogliono mettere contro e in quello subito dopo asf…",['glisteninIights'],[]
2022-02-25 02:40:39,"SOTTONE LUI🥰

 #jeru",[],['jeru']
2022-02-24 22:13:27,@lindaromanoff amo ma è fighissimo no,['lindaromanoff'],[]
2022-02-23 18:10:11,"@_anamorfosi_ 😂 Sarà insieme al mio!!!
Scherzo ma troppo, e ti auguro che stia per arrivare da te ❤️",['_anamorfosi_'],[]
2022-02-24 08:20:03,RT @mgmaglie: L'obbligo vaccinale #over50 non avrebbe mai dovuto essere pensato da un governo anche solo vagamente ispirato a principi di d…,['mgmaglie'],['over50']
2022-02-23 20:34:03,"@sscnapoli Comunque non è la grafica che a me piace, è il pullman che è strano…un vorrei ma non posso, classico stile della famiglia cinematografica romana.",['sscnapoli'],[]
2022-02-25 00:22:20,"RT @Ypila18: TRASMISSIONE VERGOGNOSA,
ALFONSO NN IN GRADO DI TENERE IL PROGRAMMA,LE SUE PROTETTE CHE SONO STATE SEMPRE SALVATE E MAI RIPRES…",['Ypila18'],[]
2022-02-24 07:57:48,RT @thevalse_: preferirei lo scudetto del milaahahha no ben venga la guerra porcodio,['thevalse_'],[]
2022-02-24 11:15:53,"RT @amendolaenzo: La Russia è l'unica responsabile di questa inaccettabile aggressione. L'Italia, la UE, la Nato, il mondo libero, tutti no…",['amendolaenzo'],[]
2022-02-23 18:52:27,"@Giandom84354994 @tiecolino ""Solidarietà dal PD"".
Questo mi offenderebbe davvero...","['Giandom84354994', 'tiecolino']",[]
2022-02-23 23:01:00,ahaaha,[],[]
2022-02-24 11:02:12,avevo un brutto presentimento e infatti once again ambulanza davanti a casa i Hope hes ok,[],[]
2022-02-25 05:18:38,"RT @TgLa7: #RussiaUkraineConflict
Le forze armate russe entrate in Ucraina dalla Bielorussia sarebbero a circa 30 chilometri da Kiev. L'obi…",['TgLa7'],['RussiaUkraineConflict']
2022-02-24 21:32:29,Albe sotto i post di matti🖤 https://t.co/sNyfoSNBtS,[],[]
2022-02-23 19:35:12,Aggiornamento serale. Siamo vicini a quota 30.000 biglietti venduti.,[],[]
2022-02-23 20:14:08,"RT @Moonlightshad1: Se si è potuta replicare la stessa situazione di ""Ruby è la nipote di Mubarak"" è perché la prima volta dopo non è succe…",['Moonlightshad1'],[]
2022-02-23 17:39:07,"Pnrr, Nardella a Draghi: ""Le città vero motore per il successo del Piano"" https://t.co/ZiIViA1CCx",[],[]
2022-02-24 21:28:58,@esauritatorino Quel mio gesto grande di amore per te,['esauritatorino'],[]
2022-02-23 17:49:49,Incredibile come T faccia riferimento al pollo dell'ess3lung4 nel video,[],[]
2022-02-24 16:24:22,Era nel 1995... https://t.co/aTRHTfK3zs,[],[]
2022-02-24 17:31:45,@wydDeezy come ti trovi adesso,['wydDeezy'],[]
2022-02-23 23:30:41,y yo con un filtro raro pero chulo sjjsjsj https://t.co/AYmAINxfl4,[],[]
2022-02-23 22:29:28,purtroppo a noi,[],[]
2022-02-25 01:34:40,"RT @ovexagain: 🚨 FATE GIRARE🚨

#jeru #gfvip #jessvip #basciagoni",['ovexagain'],"['jeru', 'gfvip', 'jessvip', 'basciagoni']"
2022-02-24 17:55:14,"RT @youcancallmegiu: Se usare i social per parlare della situazione diventa troppo pesante per voi, non sentitevi in colpa se usate i socia…",['youcancallmegiu'],[]
2022-02-24 16:45:33,Il sonno della ragione,[],[]
2022-02-25 01:40:47,@assymoccia1998 La coda di paglia….,['assymoccia1998'],[]
2022-02-24 07:59:57,"Ha parlato pure la signora: 
""agiamo di concerto quando possiamo, da soli quando vogliamo, esiste un posticino particolare all' inferno per le donne che non hanno Clinton"". 
Daje Maddeleme facce Tarzan.",[],[]

### Mongo example JSON format 


In [1]:
witter> db.QCPS_2.aggregate([{ $sample: { size: 8 } }])
[
  {
    _id: ObjectId('621821374330c45866af2402'),
    in_reply_to_status_id: -1,
    possibly_sensitive: false,
    created_at: ISODate('2022-02-25T00:22:03.000Z'),
    truncated: true,
    source: '<a href="http://twitter.com/download/iphone" rel="nofollow">Twitter for iPhone</a>',
    retweet_count: 0,
    hashtagEntities: 'basciagoni|jessvip|miriangeles|fairylulu|jerù',
    favourited_count: 0,
    in_reply_to_screen_name: null,
    in_reply_to_user_id: -1,
    id: Long('1497003926335406083'),
    text: 'Basciano sta ritirando tutto perchè ha paura delle diffide. Capite che se non ci sono prove, non vuole aizzare la situazione. Noi sappiamo la verità! #basciagoni #jessvip #miriangeles #fairylulu #jerù',
    hashtagEntitiesArray: [ 'basciagoni', 'jessvip', 'miriangeles', 'fairylulu', 'jerù' ],
    user: {
      utc_offset: -1,
      friends_count: 14,
      listed_count: 0,
      favourites_count: 7758,
      verified: false,
      description: 'Qui per sostenere le persone giuste',
      created_at: ISODate('2014-03-16T19:16:21.000Z'),
      time_zone: null,
      url: null,
      screen_name: 'spillthethe',
      statuses_count: 7855,
      followers_count: 52,
      name: 'Jess cuore di panna',
      location: null,
      id: Long('2415893763'),
      geo_enabled: false,
      lang: null
    },
    favorited: false
  },
  {
    _id: ObjectId('6217772c4330c45866989abe'),
    in_reply_to_status_id: -1,
    possibly_sensitive: false,
    created_at: ISODate('2022-02-24T12:16:35.000Z'),
    truncated: true,
    source: '<a href="http://twitter.com/download/iphone" rel="nofollow">Twitter for iPhone</a>',
    retweet_count: 0,
    favourited_count: 0,
    in_reply_to_screen_name: null,
    in_reply_to_user_id: -1,
    id: Long('1496821355576242180'),
    text: 'cercare di distrarsi non è assolutamente sinonimo di menefreghismo, vi prego smettetela di ragionare in questo modo, ognuno affronta la situazione come se la sente',
    user: {
      utc_offset: -1,
      friends_count: 462,
      listed_count: 1,
      favourites_count: 55031,
      verified: false,
      description: 'ceo of croccantelle al ketchup',
      created_at: ISODate('2021-03-24T15:38:32.000Z'),
      time_zone: null,
      url: null,
      screen_name: 'ssheisalady',
      statuses_count: 10082,
      followers_count: 493,
      name: 'val🌱',
      location: 'she/her',
      id: Long('1374747434707001346'),
      geo_enabled: false,
      lang: null
    },
    favorited: false
  },
  {
    _id: ObjectId('6217570b4330c45866950680'),
    in_reply_to_status_id: -1,
    possibly_sensitive: false,
    userMentionEntities: 'ProItalia_org',
    created_at: ISODate('2022-02-24T09:59:08.000Z'),
    truncated: false,
    source: '<a href="https://mobile.twitter.com" rel="nofollow">Twitter Web App</a>',
    retweeted_status: {
      in_reply_to_status_id: -1,
      possibly_sensitive: false,
      created_at: ISODate('2022-02-23T23:38:04.000Z'),
      truncated: true,
      source: '<a href="http://twitter.com/download/android" rel="nofollow">Twitter for Android</a>',
      retweet_count: 47,
      hashtagEntities: 'BASTAgreenpass',
      favourited_count: 130,
      in_reply_to_screen_name: null,
      in_reply_to_user_id: -1,
      id: Long('1496630468900532227'),
      text: 'Prima vi mettono in una prigione di 4 mq. Poi in una di 2 mq. E infine di nuovo in una di 4mq, dicendovi che vi hanno liberati.\n' +
        'E voi applaudite?\n' +
        '#BASTAgreenpass',
      hashtagEntitiesArray: [ 'BASTAgreenpass' ],
      user: {
        utc_offset: -1,
        friends_count: 310,
        listed_count: 7,
        favourites_count: 1323,
        verified: false,
        description: "Per un'Italia libera, forte e prospera, all'altezza di se stessa e proiettata nel futuro.",
        created_at: ISODate('2021-11-01T16:07:34.000Z'),
        time_zone: null,
        url: 'http://www.proitalia.org',
        screen_name: 'ProItalia_org',
        statuses_count: 1083,
        followers_count: 2636,
        name: 'Pro Italia',
        location: null,
        id: Long('1455204821242372096'),
        geo_enabled: false,
        lang: null
      },
      favorited: false
    },
    retweet_count: 0,
    favourited_count: 0,
    in_reply_to_screen_name: null,
    userMentionEntitiesArray: [ 'ProItalia_org' ],
    in_reply_to_user_id: -1,
    id: Long('1496786764794585093'),
    text: 'RT @ProItalia_org: Prima vi mettono in una prigione di 4 mq. Poi in una di 2 mq. E infine di nuovo in una di 4mq, dicendovi che vi hanno li…',
    user: {
      utc_offset: -1,
      friends_count: 6674,
      listed_count: 16,
      favourites_count: 48786,
      verified: false,
      description: "Per vedere di nascosto l'effetto che fa - Ironia e sarcasmo q.b. - Ne cives ad arma ruant - #NoGreenPass",
      created_at: ISODate('2018-10-28T08:44:58.000Z'),
      time_zone: null,
      url: null,
      screen_name: 'Luca__Pagani',
      statuses_count: 17686,
      followers_count: 9366,
      name: 'Luca Pagani',
      location: 'Italia',
      id: Long('1056466845295935488'),
      geo_enabled: false,
      lang: null
    },
    favorited: false
  },
  {
    _id: ObjectId('621780d14330c4586699e073'),
    in_reply_to_status_id: -1,
    possibly_sensitive: false,
    userMentionEntities: 'SamTonellato',
    created_at: ISODate('2022-02-24T12:57:26.000Z'),
    truncated: false,
    source: '<a href="http://twitter.com/download/android" rel="nofollow">Twitter for Android</a>',
    retweeted_status: {
      in_reply_to_status_id: -1,
      possibly_sensitive: false,
      created_at: ISODate('2022-02-24T12:33:48.000Z'),
      truncated: false,
      source: '<a href="http://twitter.com/download/iphone" rel="nofollow">Twitter for iPhone</a>',
      retweet_count: 5,
      hashtagEntities: 'fairylu|GFvip',
      favourited_count: 63,
      in_reply_to_screen_name: null,
      in_reply_to_user_id: -1,
      id: Long('1496825687268855812'),
      text: 'dimmi che rosichi ancora che lulù sia andata finale senza dirmelo…inizia katia 🌚 #fairylu #GFvip https://t.co/R8is14HldT',
      hashtagEntitiesArray: [ 'fairylu', 'GFvip' ],
      user: {
        utc_offset: -1,
        friends_count: 202,
        listed_count: 7,
        favourites_count: 13222,
        verified: false,
        description: '•cyrus_grande_kordei• IG: www_uomonero_it 🇱🇰•🇧🇷',
        created_at: ISODate('2013-11-05T18:22:38.000Z'),
        time_zone: null,
        url: 'https://www.instagram.com/www_uomonero_it/',
        screen_name: 'SamTonellato',
        statuses_count: 2955,
        followers_count: 871,
        name: 'sam (uomonero) 👼🏾',
        location: 'Venezia, Veneto',
        id: Long('2176572565'),
        geo_enabled: true,
        lang: null
      },
      favorited: false
    },
    retweet_count: 0,
    hashtagEntities: 'fairylu|GFvip',
    favourited_count: 0,
    in_reply_to_screen_name: null,
    userMentionEntitiesArray: [ 'SamTonellato' ],
    in_reply_to_user_id: -1,
    id: Long('1496831635794411523'),
    text: 'RT @SamTonellato: dimmi che rosichi ancora che lulù sia andata finale senza dirmelo…inizia katia 🌚 #fairylu #GFvip https://t.co/R8is14HldT',
    hashtagEntitiesArray: [ 'fairylu', 'GFvip' ],
    user: {
      utc_offset: -1,
      friends_count: 14,
      listed_count: 0,
      favourites_count: 8797,
      verified: false,
      description: null,
      created_at: ISODate('2021-11-22T19:34:01.000Z'),
      time_zone: null,
      url: null,
      screen_name: 'AlbachiaraPari1',
      statuses_count: 1269,
      followers_count: 5,
      name: 'Albachiara Parisi',
      location: null,
      id: Long('1462866762647539712'),
      geo_enabled: false,
      lang: null
    },
    favorited: false
  },
  {
    _id: ObjectId('62175b534330c458669571ac'),
    in_reply_to_status_id: -1,
    possibly_sensitive: false,
    userMentionEntities: 'valigiablu|valigiablu',
    created_at: ISODate('2022-02-24T10:17:38.000Z'),
    truncated: false,
    source: '<a href="http://twitter.com/download/android" rel="nofollow">Twitter for Android</a>',
    retweeted_status: {
      in_reply_to_status_id: -1,
      possibly_sensitive: false,
      userMentionEntities: 'valigiablu',
      created_at: ISODate('2022-02-24T10:14:52.000Z'),
      truncated: true,
      source: '<a href="https://mobile.twitter.com" rel="nofollow">Twitter Web App</a>',
      retweet_count: 13,
      favourited_count: 23,
      in_reply_to_screen_name: null,
      userMentionEntitiesArray: [ 'valigiablu' ],
      urlEntities: 'https://www.valigiablu.it/crisi-russia-ucraina-disinformazione/',
      in_reply_to_user_id: -1,
      urlEntitiesArray: [
        'https://www.valigiablu.it/crisi-russia-ucraina-disinformazione/'
      ],
      id: Long('1496790724725157890'),
      text: 'Crisi Russia-Ucraina: come seguire le notizie in tempo reale in modo responsabile https://t.co/yW9Xgf9vgm via @valigiablu https://t.co/5MGFvT8ERX',
      user: {
        utc_offset: -1,
        friends_count: 1204,
        listed_count: 780,
        favourites_count: 14886,
        verified: true,
        description: 'Basata sui fatti. Aperta a tutti. Sostenuta dai lettori. Partecipa al crowdfunding per sostenere Valigia Blu: http://crowdfunding.valigiablu.it',
        created_at: ISODate('2010-05-21T13:21:21.000Z'),
        time_zone: null,
        url: 'http://www.valigiablu.it',
        screen_name: 'valigiablu',
        statuses_count: 36127,
        followers_count: 91173,
        name: 'Valigia Blu',
        location: null,
        id: 146448681,
        geo_enabled: false,
        lang: null
      },
      favorited: false
    },
    retweet_count: 0,
    favourited_count: 0,
    in_reply_to_screen_name: null,
    userMentionEntitiesArray: [ 'valigiablu', 'valigiablu' ],
    urlEntities: 'https://www.valigiablu.it/crisi-russia-ucraina-disinformazione/',
    in_reply_to_user_id: -1,
    urlEntitiesArray: [
      'https://www.valigiablu.it/crisi-russia-ucraina-disinformazione/'
    ],
    id: Long('1496791421650620418'),
    text: 'RT @valigiablu: Crisi Russia-Ucraina: come seguire le notizie in tempo reale in modo responsabile https://t.co/yW9Xgf9vgm via @valigiablu h…',
    user: {
      utc_offset: -1,
      friends_count: 181,
      listed_count: 0,
      favourites_count: 1796,
      verified: false,
      description: 'Scienze politiche @unipisa🏛️ \n' +
        'Morte a Videodrome. Lunga vita alla nuova carne.',
      created_at: ISODate('2020-11-28T09:10:29.000Z'),
      time_zone: null,
      url: null,
      screen_name: 'FraGiova1',
      statuses_count: 1073,
      followers_count: 16,
      name: 'Francesco Giovacchini',
      location: 'Lucca, Toscana',
      id: Long('1332612781632315393'),
      geo_enabled: true,
      lang: null
    },
    favorited: false
  },
  {
    _id: ObjectId('6217c67a4330c45866a2af00'),
    in_reply_to_status_id: -1,
    possibly_sensitive: false,
    userMentionEntities: 'Radio1Rai',
    created_at: ISODate('2022-02-24T17:54:42.000Z'),
    truncated: false,
    source: '<a href="http://twitter.com/download/android" rel="nofollow">Twitter for Android</a>',
    retweeted_status: {
      in_reply_to_status_id: -1,
      possibly_sensitive: false,
      userMentionEntities: 'santa_cecilia',
      created_at: ISODate('2022-02-24T16:55:01.000Z'),
      truncated: true,
      source: '<a href="https://studio.twitter.com" rel="nofollow">Twitter Media Studio</a>',
      retweet_count: 3,
      favourited_count: 3,
      in_reply_to_screen_name: null,
      userMentionEntitiesArray: [ 'santa_cecilia' ],
      in_reply_to_user_id: -1,
      id: Long('1496891426080235526'),
      text: "🔵 Il direttore milanese Daniele Gatti torna da stasera a Roma sul podio dell'orchestra dell'Accademia Nazionale di Santa Cecilia @santa_cecilia\n" +
        `per una "lettura" di due capolavori sinfonici dell'Ottocento. L'intervista di Claudia Fayenz https://t.co/scjzJykVZs`,
      user: {
        utc_offset: -1,
        friends_count: 496,
        listed_count: 754,
        favourites_count: 13734,
        verified: true,
        description: 'La tua informazione, sempre',
        created_at: ISODate('2010-10-31T15:50:05.000Z'),
        time_zone: null,
        url: 'https://www.raiplayradio.it/radio1',
        screen_name: 'Radio1Rai',
        statuses_count: 156795,
        followers_count: 68139,
        name: 'Rai Radio1',
        location: 'Rome, Lazio',
        id: 210501383,
        geo_enabled: true,
        lang: null
      },
      favorited: false
    },
    retweet_count: 0,
    favourited_count: 0,
    in_reply_to_screen_name: null,
    userMentionEntitiesArray: [ 'Radio1Rai' ],
    in_reply_to_user_id: -1,
    id: Long('1496906447430504455'),
    text: "RT @Radio1Rai: 🔵 Il direttore milanese Daniele Gatti torna da stasera a Roma sul podio dell'orchestra dell'Accademia Nazionale di Santa Cec…",
    user: {
      utc_offset: -1,
      friends_count: 1495,
      listed_count: 1,
      favourites_count: 24929,
      verified: false,
      description: null,
      created_at: ISODate('2021-02-28T18:55:12.000Z'),
      time_zone: null,
      url: null,
      screen_name: 'Alsaleh23357005',
      statuses_count: 20682,
      followers_count: 71,
      name: 'Alsaleh',
      location: 'الرياض ',
      id: Long('1366099459415945217'),
      geo_enabled: false,
      lang: null
    },
    favorited: false
  },
  {
    _id: ObjectId('62179d854330c458669d9e3e'),
    in_reply_to_status_id: -1,
    possibly_sensitive: false,
    userMentionEntities: 'sailor_snickers',
    created_at: ISODate('2022-02-24T15:00:02.000Z'),
    truncated: false,
    source: '<a href="http://twitter.com/download/android" rel="nofollow">Twitter for Android</a>',
    retweeted_status: {
      in_reply_to_status_id: -1,
      possibly_sensitive: false,
      created_at: ISODate('2022-02-24T10:03:12.000Z'),
      truncated: true,
      source: '<a href="http://twitter.com/download/iphone" rel="nofollow">Twitter for iPhone</a>',
      retweet_count: 126,
      favourited_count: 964,
      in_reply_to_screen_name: null,
      in_reply_to_user_id: -1,
      id: Long('1496787788867514371'),
      text: 'Ma chi se ne fotte di chi ha ragione o meno, durante le guerre a morire sono sempre e solo le persone innocenti e questo dovrebbe far inorridire tutti, a prescindere',
      user: {
        utc_offset: -1,
        friends_count: 2518,
        listed_count: 54,
        favourites_count: 211714,
        verified: false,
        description: 'In realtà sono un uomo di 40 anni che vive nel vostro seminterrato.',
        created_at: ISODate('2016-02-15T17:34:01.000Z'),
        time_zone: null,
        url: 'https://www.instagram.com/starfly__93/?hl=it',
        screen_name: 'sailor_snickers',
        statuses_count: 174750,
        followers_count: 20259,
        name: 'Sailor Snickers ☾',
        location: 'Napoli, Campania',
        id: Long('4914670317'),
        geo_enabled: true,
        lang: null
      },
      favorited: false
    },
    retweet_count: 0,
    favourited_count: 0,
    in_reply_to_screen_name: null,
    userMentionEntitiesArray: [ 'sailor_snickers' ],
    in_reply_to_user_id: -1,
    id: Long('1496862488759050246'),
    text: 'RT @sailor_snickers: Ma chi se ne fotte di chi ha ragione o meno, durante le guerre a morire sono sempre e solo le persone innocenti e ques…',
    user: {
      utc_offset: -1,
      friends_count: 427,
      listed_count: 0,
      favourites_count: 102477,
      verified: false,
      description: "i'm so afraid of losing something i love that i refuse to love anything",
      created_at: ISODate('2015-09-20T14:11:33.000Z'),
      time_zone: null,
      url: null,
      screen_name: 'fede_lxl',
      statuses_count: 7226,
      followers_count: 345,
      name: 'federica 🌻',
      location: 'she/her',
      id: Long('3718340837'),
      geo_enabled: false,
      lang: null
    },
    favorited: false
  },
  {
    _id: ObjectId('62166d7f4330c4586683596d'),
    in_reply_to_status_id: -1,
    possibly_sensitive: false,
    created_at: ISODate('2022-02-23T17:23:02.000Z'),
    truncated: false,
    source: '<a href="http://twitter.com/download/iphone" rel="nofollow">Twitter for iPhone</a>',
    retweet_count: 0,
    favourited_count: 0,
    in_reply_to_screen_name: null,
    in_reply_to_user_id: -1,
    id: Long('1496536087954665475'),
    text: 'Accurate info 😂😂😂',
    user: {
      utc_offset: -1,
      friends_count: 4971,
      listed_count: 76,
      favourites_count: 4694,
      verified: false,
      description: 'Lobbyist,Cannabis Entrepreneur,Fmr State Representative 179th District PA 2006-2012, Vertical Farming enthusiast',
      created_at: ISODate('2010-09-29T15:05:07.000Z'),
      time_zone: null,
      url: 'http://www.davidscottpartners.com',
      screen_name: 'TonyPaytonJr',
      statuses_count: 6961,
      followers_count: 3675,
      name: 'Tony Payton Jr.',
      location: 'Philadelphia, PA 19102',
      id: 196625354,
      geo_enabled: false,
      lang: null
    },
    favorited: false
  }
]
twitter> 

SyntaxError: invalid syntax (1152424647.py, line 1)

### More Mongo JSON example 

*!remember that in mongo they are binary JSON!* 

In [2]:
[{
    _id: ObjectId('6217acbb4330c458669f9f9f'),
    in_reply_to_status_id: -1,
    possibly_sensitive: false,
    userMentionEntities: 'TeaTeaser',
    created_at: ISODate('2022-02-24T16:05:06.000Z'),
    truncated: false,
    source: '<a href="http://twitter.com/download/android" rel="nofollow">Twitter for Android</a>',
    retweeted_status: {
      in_reply_to_status_id: -1,
      possibly_sensitive: false,
      created_at: ISODate('2022-02-23T20:56:46.000Z'),
      truncated: true,
      source: '<a href="http://twitter.com/download/android" rel="nofollow">Twitter for Android</a>',
      retweet_count: 197,
      favourited_count: 331,
      in_reply_to_screen_name: null,
      in_reply_to_user_id: -1,
      id: Long('1496589877844054026'),
      text: 'Il Presidente della squadra di calcio dello Steaua Bucarest Gigi Becali non vuole giocatori "vaccinati" perché i giocatori vaccinati perdono forza. https://t.co/ofFzaR0DzD',
      user: {
        utc_offset: -1,
        friends_count: 248,
        listed_count: 1,
        favourites_count: 2334,
        verified: false,
        description: 'Se il sistema è ingiusto e corrotto è mio dovere combatterlo. La logica e la coerenza sono le chiavi per arrivare alla verità. Chi mi segue verrà seguito.',
        created_at: ISODate('2013-04-06T20:11:29.000Z'),
        time_zone: null,
        url: null,
        screen_name: 'TeaTeaser',
        statuses_count: 1849,
        followers_count: 177,
        name: 'STOP The Great Reset',
        location: null,
        id: 1332401534,
        geo_enabled: false,
        lang: null
      },
      favorited: false
    },
    retweet_count: 0,
    favourited_count: 0,
    in_reply_to_screen_name: null,
    userMentionEntitiesArray: [ 'TeaTeaser' ],
    in_reply_to_user_id: -1,
    id: Long('1496878862990548997'),
    text: 'RT @TeaTeaser: Il Presidente della squadra di calcio dello Steaua Bucarest Gigi Becali non vuole giocatori "vaccinati" perché i giocatori v…',
    user: {
      utc_offset: -1,
      friends_count: 892,
      listed_count: 20,
      favourites_count: 12578,
      verified: false,
      description: '1979\nAmministratore della Akhet caffè',
      created_at: ISODate('2013-02-26T15:57:40.000Z'),
      time_zone: null,
      url: null,
      screen_name: 'veratto_A',
      statuses_count: 85800,
      followers_count: 591,
      name: 'V. alessandro',
      location: null,
      id: 1222227577,
      geo_enabled: false,
      lang: null
    },
    favorited: false
  },
  {
    _id: ObjectId('6216a1744330c458668a066a'),
    in_reply_to_status_id: Long('1496590525138448391'),
    possibly_sensitive: false,
    userMentionEntities: 'lina_palmese|redazioneiene|Italia1',
    created_at: ISODate('2022-02-23T21:04:28.000Z'),
    truncated: false,
    source: '<a href="http://twitter.com/download/iphone" rel="nofollow">Twitter for iPhone</a>',
    retweet_count: 0,
    hashtagEntities: 'SCANUALLEIENE',
    favourited_count: 0,
    in_reply_to_screen_name: 'lina_palmese',
    userMentionEntitiesArray: [ 'lina_palmese', 'redazioneiene', 'Italia1' ],
    in_reply_to_user_id: 422823097,
    id: Long('1496591814584901632'),
    text: '@lina_palmese @redazioneiene @Italia1 Apri tutte le porteeee e fai entrare il sooleee #SCANUALLEIENE https://t.co/sTmkNOY5xB',
    hashtagEntitiesArray: [ 'SCANUALLEIENE' ],
    user: {
      utc_offset: -1,
      friends_count: 1174,
      listed_count: 2,
      favourites_count: 9624,
      verified: false,
      description: "• Farò sempre musica, in un modo o nell'altro • ❤",
      created_at: ISODate('2012-03-14T14:49:32.000Z'),
      time_zone: null,
      url: null,
      screen_name: 'MartaGobatto',
      statuses_count: 22600,
      followers_count: 960,
      name: 'MartaGobatto',
      location: null,
      id: 524420458,
      geo_enabled: true,
      lang: null
    },
    favorited: false
  }
]
 {
    _id: ObjectId('6217a9ba4330c458669f3bae'),
    in_reply_to_status_id: -1,
    possibly_sensitive: false,
    userMentionEntities: 'Freud2912Maury',
    created_at: ISODate('2022-02-24T15:51:59.000Z'),
    truncated: false,
    source: '<a href="http://twitter.com/download/iphone" rel="nofollow">Twitter for iPhone</a>',
    retweeted_status: {
      in_reply_to_status_id: -1,
      possibly_sensitive: false,
      created_at: ISODate('2022-02-24T09:38:36.000Z'),
      truncated: false,
      source: '<a href="http://twitter.com/download/iphone" rel="nofollow">Twitter for iPhone</a>',
      retweet_count: 24,
      favourited_count: 34,
      in_reply_to_screen_name: null,
      in_reply_to_user_id: -1,
      id: Long('1496781599924535298'),
      text: 'Quanti atti mancati nella vita che paghiamo a caro prezzo.',
      user: {
        utc_offset: -1,
        friends_count: 215,
        listed_count: 2,
        favourites_count: 13090,
        verified: false,
        description: 'Certi giorni provi a capire, altri puoi solo resistere.',
        created_at: ISODate('2021-09-01T11:23:40.000Z'),
        time_zone: null,
        url: null,
        screen_name: 'Freud2912Maury',
        statuses_count: 9196,
        followers_count: 656,
        name: 'Maury',
        location: 'Napoli',
        id: Long('1433027634758307841'),
        geo_enabled: false,
        lang: null
      },
      favorited: false
    },
    retweet_count: 0,
    favourited_count: 0,
    in_reply_to_screen_name: null,
    userMentionEntitiesArray: [ 'Freud2912Maury' ],
    in_reply_to_user_id: -1,
    id: Long('1496875564447121412'),
    text: 'RT @Freud2912Maury: Quanti atti mancati nella vita che paghiamo a caro prezzo.',
    user: {
      utc_offset: -1,
      friends_count: 893,
      listed_count: 17,
      favourites_count: 210166,
      verified: false,
      description: 'E hai preso a pugni tutte le nostre incertezze, avessi forza per arrendermi io lo farei; @MarroneEmma @LDAilvero @colesprouse || Carola Puddu •EXIT🪁 esserequi',
      created_at: ISODate('2014-03-20T14:11:10.000Z'),
      time_zone: null,
      url: null,
      screen_name: 'amorsisulcuore',
      statuses_count: 285502,
      followers_count: 3549,
      name: 'Valeria E.',
      location: 'Roma',
      id: Long('2424790209'),
      geo_enabled: false,
      lang: null
    },
    favorited: false
  }

IndentationError: unexpected indent (3633937033.py, line 109)